In [1]:
import torch
import torch.nn as nn

device = "cuda"

In [ ]:
from torch import Tensor

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels:int, embed_dim:int, patch_size=16):
        super().__init__()
        self.embed = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, X:Tensor): #input dim [batch, seq, h, w]
        X = self.embed(X)
        X = X.flatten(2)
        return X.transpose(1,2)

class vit(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channel:int=3,
                 num_classes:int=1000, embed_dim=768, depth:int=12, num_heads:int=12,
                 ff_dim:int=3072, dropout:float=0.1):
        super().__init__()
        self.embed = PatchEmbedding(in_channel, embed_dim, patch_size)
        cls_init = torch.randn(1,1,embed_dim) * 0.02
        self.cls_token = nn.Parameter(cls_init)
        num_patches = (img_size // patch_size) ** 2
        pos_init = torch.randn(1, num_patches + 1, embed_dim) * 0.02
        self.pos_embed = nn.Parameter(pos_init)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            embed_dim, num_heads, ff_dim, dropout, "gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, depth)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.output = nn.Linear(embed_dim, num_classes)
    
    def forward(self, X:Tensor):
        Z = self.embed(X)
        
        Z = torch.concat([self.cls_token, Z]) + self.pos_embed
        Z = Z + self.pos_embed
        Z = self.dropout(Z)
        Z = self.encoder(Z)
        Z = self.layer_norm(Z[:, 0])
        
        logits = self.output(Z)
        return logits

SyntaxError: incomplete input (3792535346.py, line 7)